## 2. Define Helper Functions to Parse Matrix Files

deal.II's `print_as_numpy_arrays()` outputs matrices in a format with three arrays: row indices, column indices, and values.

In [113]:
import numpy as np
import scipy.sparse as sp
from pathlib import Path


## 1. Import Required Libraries

In [114]:
def read_dealii_matrix(filename):
    """
    Read a matrix file generated by deal.II's print_as_numpy_arrays().
    
    Returns a scipy sparse matrix in CSR format.
    """
    try:
        # Load the three arrays directly using numpy
        [data, row, column] = np.loadtxt(filename)
        
        # Convert to integers for row and column indices
        row = row.astype(int)
        column = column.astype(int)
        
        # Create sparse matrix in CSR format
        sparse_matrix = sp.csr_matrix((data, (row, column)))
        
        print(f"Successfully loaded matrix from {filename}")
        print(f"  Shape: {sparse_matrix.shape}")
        print(f"  Non-zero entries: {sparse_matrix.nnz}")
        
        return sparse_matrix
    
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None

# Matrix Analysis for continuous agglomeration multigrid

This notebook reads and analyzes sparse matrices generated by deal.II's `print_as_numpy_arrays()` function from the continuous Poisson solver code.

The matrices analyzed are:
- **System Matrix**: The finest level system matrix from the Poisson problem
- **Transfer Matrix**: The agglomeration to original triangulation transfer matrix

In [115]:
system_matrix =read_dealii_matrix("build/system_matrix.txt")
transfer_matrix = read_dealii_matrix("build/transfer_matrix_agglo_to_original_tria.txt")
transfer_matrix_agglo_to_agglo = read_dealii_matrix("build/transfer_matrix_agglo_to_agglo.txt")

#show the non zero entries of transfer matrix
print("Non-zero entries of transfer matrix:", transfer_matrix.nnz)
#Shown the min and max entries of transfer matrix
# print("Min entry of transfer matrix:", transfer_matrix.data.min())
# print("Max entry of transfer matrix:", transfer_matrix.data.max())

Successfully loaded matrix from build/system_matrix.txt
  Shape: (289, 289)
  Non-zero entries: 2401
Successfully loaded matrix from build/transfer_matrix_agglo_to_original_tria.txt
  Shape: (289, 148)
  Non-zero entries: 1156
Successfully loaded matrix from build/transfer_matrix_agglo_to_agglo.txt
  Shape: (148, 20)
  Non-zero entries: 592
Non-zero entries of transfer matrix: 1156


In [116]:
#Print condition number of original matrix
print("Condition number of system matrix:",np.linalg.cond(system_matrix.toarray()))
#Check it is SPD
eigenvalues = np.linalg.eigvalsh(system_matrix.toarray())
if np.all(eigenvalues > 0):
    print("The system matrix is symmetric positive definite (SPD).")
else:
    print("The system matrix is NOT symmetric positive definite (SPD).")


Condition number of system matrix: 51.71439460456186
The system matrix is symmetric positive definite (SPD).


In [117]:
#now compute the triple product P^T A P
triple_product = transfer_matrix.T @ system_matrix @ transfer_matrix
print("Triple product shape:", triple_product.shape)
eigenvalues = np.linalg.eigvalsh(triple_product.toarray())
if np.all(eigenvalues > 0):
    print("The induced matrix is symmetric positive definite (SPD).")
else:
    print("The induced matrix is NOT symmetric positive definite (SPD).")


Triple product shape: (148, 148)
The induced matrix is symmetric positive definite (SPD).


In [112]:
#now compute the triple product P^T A P with agglo to agglo transfer and A induced matrix above
triple_product_agglo = transfer_matrix_agglo_to_agglo.T @ triple_product @ transfer_matrix_agglo_to_agglo
print("Triple product shape:", triple_product_agglo.shape)
eigenvalues = np.linalg.eigvalsh(triple_product_agglo.toarray())
if np.all(eigenvalues > 0):
    print("The induced matrix is symmetric positive definite (SPD).")
else:
    print("The induced matrix is NOT symmetric positive definite (SPD).")


Triple product shape: (20, 20)
The induced matrix is symmetric positive definite (SPD).
